In [ ]:
#Inspired by https://avandekleut.github.io/vae/
import torch;
import torch.nn as nn
import torch.utils
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import wandb
wandb.login()        

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
class FF(nn.Module):
    def __init__(self,dim1,dim2,dim3):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(in_features=dim1, out_features=dim2),
            nn.ReLU(),
            nn.Linear(in_features=dim2, out_features=dim3)
        )

    def forward(self, input):
        return self.main(input)
tmp = FF(28*28,512,2)
print(tmp)
print(tmp(torch.rand(10,1,28*28)).shape)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, dim1, dim2, dim3):
        super().__init__()
        self.encoder = FF(dim1, dim2, dim3)
        self.decoder = nn.Sequential(
            FF(dim3, dim2, dim1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x
tmp = Autoencoder(28*28,512,2)
print(tmp)
print(tmp(torch.rand(10,1,28*28)).shape)

In [ ]:
def train(data_loader, model, optimizer, loss_function, epochs=20):
    model.to(device)
    losses = []
    global_step = 0

    for epoch in range(epochs):
        for i, (x, y) in enumerate(data_loader):
            x = x.to(device)

            optimizer.zero_grad()
            x_hat = model(x)
            loss = loss_function(x, x_hat)

            losses.append(loss.item())
            loss.backward()
            optimizer.step()

            # log to W&B
            wandb.log({"train_loss": loss.item(), "epoch": epoch}, step=global_step)
            global_step += 1

            if i % 100 == 0:
                print(f"{epoch=}/{i}: {loss}")
    return model, losses

In [ ]:
def plot_latent(data_loader, encoder, dim1=0, dim2=1, num_batches=100):
    for i, (x, y) in enumerate(data_loader):
        z = encoder(x.to(device))
        z = z.to('cpu').detach().numpy()
        plt.scatter(z[:, 0, dim1], z[:, 0, dim2], c=y, alpha=0.5)
        if i > num_batches:
            plt.colorbar()
            break

In [ ]:
def plot_reconstructed(decoder, w, h, r0=(-10, 10), r1=(-10, 10), n=12):
    img = np.zeros((n*w, n*h))
    for i, y in enumerate(np.linspace(*r1, n)):
        for j, x in enumerate(np.linspace(*r0, n)):
            z = torch.Tensor([[x, y]]).view(1,1,2).to(device)
            x_hat = decoder(z)
            x_hat = x_hat.reshape(w, h).to('cpu').detach().numpy()
            img[(n-1-i)*w:(n-1-i+1)*w, j*w:(j+1)*w] = x_hat
    plt.imshow(img, extent=[*r0, *r1])

In [ ]:
# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.flatten(x,start_dim=-2))
])

data = datasets.MNIST('./data',transform=transform,download=True)

n, w, h = data.data.shape

data_loader = torch.utils.data.DataLoader(data,batch_size=128,shuffle=True)

LATENT_DIM = 9
HIDDEN_DIM = 512
EPOCHS = 20
BATCH_SIZE = 128

run = wandb.init(
    project="mnist-autoencoder",
    config=dict(
        latent_dim=LATENT_DIM,
        hidden_dim=HIDDEN_DIM,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
    )
)


model = Autoencoder(w * h, HIDDEN_DIM, LATENT_DIM)
optimizer = torch.optim.Adam(model.parameters())
loss_function = torch.nn.MSELoss()

(autoencoder, losses) = train(data_loader, model, optimizer, loss_function, EPOCHS)

In [ ]:
# Defining the Plot Style
plt.xlabel('Iterations')
plt.ylabel('Loss')

# Plotting the losses
plt.plot(losses)
plt.savefig("loss_curve.png")
wandb.log({"loss_curve": wandb.Image(plt.gcf())})
plt.close()

In [ ]:
# Plot latent space
plot_latent(data_loader, autoencoder.encoder)
plt.savefig('latent.pdf')
wandb.log({"latent_space": wandb.Image(plt.gcf())})
plt.close()

In [ ]:
# Generate samples from latent space
try:
    if LATENT_DIM >= 2:
        plot_reconstructed(
            autoencoder.decoder,
            w,
            h,
            r0=[-3, 3],
            r1=[-3, 3],
            n=16,
            latent_dim=LATENT_DIM,
        )
    else:
        print("LATENT_DIM < 2: Cannot plot 2D latent grid.")
    plt.savefig("reconstruction.pdf")
    wandb.log({"reconstruction_grid": wandb.Image(plt.gcf())})
except Exception as e:
    print(f"Error plotting latent reconstructions: {e}")
    print(
        "If you changed LATENT_DIM, ensure plot_reconstructed uses the same latent dimension."
    )
finally:
    plt.close()

In [ ]:
wandb.finish()

## **Answers to Task 2.1** 

### 2.1.1 Autoencoder Program Walkthrough

**Overview:** The notebook starts by importing PyTorch building blocks (modules, data utilities, optimizers), torchvision for MNIST, and numpy/matplotlib for analysis. It immediately detects whether CUDA is available and stores the chosen `device`, keeping tensor placement consistent across CPU/GPU during training and visualization.

**Functions:** 

The reusable `FF` module is a two-layer MLP with a ReLU in the middle; it is the core unit for both encoder and decoder. 

`Autoencoder` chains two of these blocks: the encoder compresses each 784-dimensional flattened image into a 2-D latent code, and the decoder expands that code back to 784 features, finishing with a Sigmoid so reconstructed pixels stay in range. 

The `train` routine drives optimization by iterating over DataLoader batches, moving data to `device`, running forward passes, measuring reconstruction loss, backpropagating, and stepping Adam, with periodic loss prints every 100 mini-batches. 

At last, `plot_latent` runs the encoder on batches and scatters the resulting 2-D codes colored by digit labels, while `plot_reconstructed` sweeps a grid of latent points through the decoder to show what images different latent regions produce.


### 2.1.2 The transform function

The `transform` first converts $\texttt{MNIST PIL}$ images to float tensors scaled to $[0,1]$ via \lstinline{ToTensor}, then flattens each 28×28 grid into a 784-length vector using a \lstinline{Lambda} \lstinline{flatten} so it fits the linear layers. \texttt{MNIST} is downloaded to \texttt{./data} if missing and served by a shuffled \lstinline{DataLoader} with batch size 128, ensuring randomized mini-batches throughout training. The flatten step is essential here because the model is fully connected rather than convolutional.

**Model Definition:** In the encoder, a ReLU after the first linear layer captures non-linear structure, while the final latent layer is left linear so the 2-D space remains unconstrained and easy to visualize. The decoder mirrors this pattern: a ReLU after the latent-to-hidden projection adds expressiveness, and a final Sigmoid bounds outputs to `[0,1]`, aligning with normalized pixel intensities and the reconstruction objective.

**Training & Loss:** Training uses Adam over all parameters. Reconstruction quality is measured with mean squared error between the input and decoded image. This choice matches the Gaussian-with-shared-variance view of pixel noise; paired with inputs scaled to `[0,1]` and a Sigmoid output layer, MSE provides a simple, stable signal for learning faithful reconstructions.

**Alternative Likelihood (Beta):** To better respect data constrained to `[0,1]`, the decoder could emit Beta distribution parameters per pixel instead of a single mean. Practically, it would output two positive numbers (or a mean plus concentration mapped through `softplus` with a small epsilon) and use the Beta negative log-likelihood as the loss. Inputs are typically clamped slightly away from exact 0 or 1 to avoid log singularities. This keeps outputs in `(0,1)` while allowing pixel-wise variance to adapt, offering a more flexible likelihood than the fixed-variance Gaussian implied by MSE.